In [ ]:
from openai import OpenAI
from google.colab import userdata

In [ ]:
RED = 1
YELLOW = -1
EMPTY = 0
show = {EMPTY:"⚪️", RED: "🔴", YELLOW: "🟡"}
pieces = {EMPTY: "empty", RED: "red", YELLOW: "yellow"}
cols = "ABCDEFG"

In [ ]:
class Board():
  def __init__(self):
    self.cell = [[EMPTY for x in range(7)] for y in range(6)]
    self.player = RED
    self.winner = EMPTY

  def print_board(self):
    result = ''
    for i in range(6):
      for j in range(7):
        result += show[self.cell[i][j]]
      result += '\n'
    # print(result)
    return(result)

  def print_board_json(self):
    result = ''
    result += cols
    result += '\n'
    for i in range(6):
      for j in range(7):
        result += pieces[self.cell[i][j]] + ' '
      result += '\n'
    # print(result)
    return(result)

  def move(self, next_col):
    col = (cols.find(next_col))
    if self.cell[0][col] != EMPTY:
      raise ValueError("Illegal move")
    row = 5
    while self.cell[row][col] != EMPTY:
      row -= 1
    self.cell[row][col] = self.player
    self.print_board()
    self.player = -1 * self.player
    return self.check_win()

  def check_winning_line(self, i, j, di, dj):
    if i < 0 or j < 0 or i >= 6 or j >= 7 or self.cell[i][j] == EMPTY or self.cell[i][j] != self.player:
      return 0

    return 1 + self.check_winning_line(i + di, j + dj, di, dj)

  def check_win(self):
    directions = [[0, 1], [1, 0], [1, 1], [-1, 1]]
    for i in range(5):
      for j in range(7):
        for direction in directions:
          result = self.check_winning_line(i, j, direction[0], direction[1])
          if result >= 4:
            self.winner = self.player
    return self.winner

  def available_moves(self):
    moves = []
    for i in range(7):
      if self.cell[0][i] == EMPTY:
        moves.append(cols[i])
    return moves


In [ ]:
class Player():
  def __init__(self, model, colour):
    self.llm = OpenAI(base_url=userdata.get('GROQ_BASE_URL'), api_key=userdata.get('GROQ_API_KEY'))
    self.colour = colour
    self.model = model

  def system(self, board):
    return f'''You are {self.colour} player in this connect four game.
    This is the current game situation and you have to make the next move to win.
    {board.print_board_json()}
    return with only and only column name like- A, B, C, D, E, F, G and nothing else
    '''


  def user(self):
    return f'''Whats your next move? Availabe moves are {board.available_moves()}. Only use one of these'''


In [ ]:
player1 = Player('llama-3.1-8b-instant', RED)
player2 = Player('llama-3.1-8b-instant', YELLOW)

In [ ]:
board = Board()
i = 0
while(board.winner == EMPTY):
  i += 1
  print(board.print_board())
  move1 = player1.llm.chat.completions.create(
    model=player1.model,
    messages=[
      {"role": "system", "content": player1.system(board)},
      {"role": "user", "content": player1.user()},
    ]
  )
  print('Red moves ' + move1.choices[0].message.content)
  board.move(move1.choices[0].message.content)
  if board.winner != EMPTY:
    print("Winner is " + board.winner)
    break
  print(board.print_board())
  move2 = player2.llm.chat.completions.create(
    model=player2.model,
    messages=[
      {"role": "system", "content": player2.system(board)},
      {"role": "user", "content": player2.user()},
    ]
  )
  print('Yellow moves ' + move2.choices[0].message.content)
  board.move(move2.choices[0].message.content)


In [ ]:
board.available_moves()

In [ ]:
board.check_win()